In [ ]:
# === repo-root bootstrap ===
# Locate the repository root (the dir containing `core/`), chdir there, and put
# it on sys.path so `core`/`execution`/`analysis` import and every `shared_data/`
# path resolves regardless of where the notebook is launched from.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "core")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
assert os.path.isdir(os.path.join(_root, "core")), "repo root (with core/) not found"
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

# Classical vs. Coherent CSWAP on IBM QPUs — N=3, K=2, T=1

**Caio's idea:** replace the heavy *classical* branching logic of QPA (mid-circuit
measure → `if_test` → conditional SWAP routing) with a fully *coherent* circuit that
uses **controlled-SWAPs (Fredkin gates)** to do the routing. No mid-circuit
measurement, no classical feed-forward.

This notebook runs **both** flavours on a real IBM QPU for the smallest non-trivial
case (`N=3` registers, `K=2`, `T=1`) and compares **fidelity** and **circuit cost**.

| Flavour | What it is | Where it comes from |
|---|---|---|
| **Classical (unrolled)** | The exact building block of our standard hardware workflow: enumerate the `2^((N-1)/2)=2` measurement-outcome *paths* as static circuits and post-select in classical post-processing. No `if_test` on the device. | [`end_to_end_unrolled_pittsburgh_t1.ipynb`](../../experiments/end_to_end_hardware/end_to_end_unrolled_pittsburgh_t1.ipynb) via `core.CircuitFactory` |
| **Coherent CSWAP (Fredkin)** | One static circuit. Schur swap-test writes onto a trial ancilla; the cyclic SWAP routing is applied coherently with `X`–Fredkin–`X` controlled on that ancilla. Reserve register read out at the end. | Caio's [`swapnet_n3_compare_analysis.ipynb`](swapnet_n3_compare_analysis.ipynb) |

Both use the **same** Pauli-twirling noise emulation on the data registers (so the λ axis
means the same thing for both) and the **same** fidelity definition: fraction of shots
with the reserve register read out as `|0…0⟩`.

> **Runtime.** Defaults below submit **2 jobs** (one per flavour) of a few seconds of QPU each.
> Wall-clock is dominated by the IBM **queue**, not compute. Set `USE_HARDWARE = False` for an
> instant local smoke-test on `FakeBrisbane` before spending QPU time.

## Step 1: Imports

In [ ]:
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import XGate, RZGate

from core.circuit_factory import CircuitFactory
from core.noise_models import PauliTwirlingStrategy
from analysis.result_processor import ResultProcessor

print("Imports loaded.")

## Step 2: Configuration

Knobs are sized for a quick run. The comparison is *demonstrative*, not publication-grade —
bump `N_TWIRL` / `SHOTS` / `LAMBDAS` for tighter error bars (and more QPU time).

In [ ]:
N        = 3                 # registers (this notebook is hard-wired to the N=3 coherent design)
K        = 2                 # qubits per register (d = 2^K = 4)
T        = 1                 # QPA trial rounds
LAMBDAS  = np.round(np.linspace(0.0, 1.0, 5), 4)   # depolarizing strength sweep (5 points)
N_TWIRL  = 100               # Pauli-twirl instances per (method, lambda) -> depolarizing average
SHOTS    = 256               # shots per twirled circuit
OPT_LEVEL = 3
SEEDS_PER_PATH = 3           # multi-seed transpile, keep lowest 2Q depth

DEVICE       = "ibm_pittsburgh"   # same device family as the standard workflow
USE_HARDWARE = True               # False -> FakeBrisbane local smoke-test (no QPU, no account)

RESULTS_DIR = os.path.join("sandbox", "cswap", "compare_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

n_classical = 2 ** ((N - 1) // 2)          # unrolled paths
pubs_classical = len(LAMBDAS) * N_TWIRL * n_classical
pubs_coherent  = len(LAMBDAS) * N_TWIRL
print(f"lambdas        : {LAMBDAS.tolist()}")
print(f"classical paths: {n_classical}  -> {pubs_classical} circuits  (1 job)")
print(f"coherent       : 1 circuit       -> {pubs_coherent} circuits  (1 job)")
print(f"executions/job : ~{max(pubs_classical, pubs_coherent) * SHOTS:,}  (IBM cap is 10,000,000)")
print(f"hardware       : {USE_HARDWARE}  device={DEVICE!r}")
print(f"results dir    : {RESULTS_DIR}")

## Step 3: Backend

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as IBMSampler

if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = service.backend(DEVICE)
else:
    from qiskit_ibm_runtime.fake_provider import FakeBrisbane
    backend = FakeBrisbane()

print(f"Backend: {backend.name}  ({backend.num_qubits} qubits)")

## Step 4: Circuit builders

**Classical** circuits come straight from the project's `UnrolledStrategy` — the same code path
the standard workflow runs. **Coherent** circuits are Caio's faithful `N=3` Schur+Fredkin design.

In [ ]:
# ---- Classical (unrolled) — project standard building block ----
def build_classical(n, k, t):
    """Return (golden_circuits, metadata) for the unrolled paths (no noise yet)."""
    strat = CircuitFactory.create_strategy("unrolled", k, t, n, no_reset=False)
    strat.set_noise_strategy(None)
    data = strat.build(0.0)
    circs = [d["circuit"] for d in data]
    meta  = [{kk: vv for kk, vv in d.items() if kk != "circuit"} for d in data]
    return circs, meta

# ---- Coherent CSWAP (Fredkin) — Caio's N=3 design ----
def _trial_unitary(n, k):
    """Schur test on (R1,R2) + X-Fredkin-X cyclic rotation of all n regs, as a gate on [anc]+data."""
    qc = QuantumCircuit(1 + n * k)
    anc = 0
    regs = [list(range(1 + j * k, 1 + (j + 1) * k)) for j in range(n)]
    r1, r2 = regs[0], regs[1]
    qc.h(anc)
    for i in range(k):
        qc.cswap(anc, r1[i], r2[i])
    qc.h(anc)
    qc.x(anc)                                  # rotate iff swap-test "passed" (anc == 0)
    for row in range(k):
        for j in range(n - 1, 0, -1):
            qc.cswap(anc, regs[j][row], regs[j - 1][row])
    qc.x(anc)
    return qc.to_gate(label="Schur+Fredkin")

def build_coherent(n, k, t):
    """Single coherent circuit. Deeper trials use multi-controlled trial unitaries (ctrl_state='0'*t)."""
    qr = [QuantumRegister(k, f"R{i + 1}") for i in range(n)]
    cr = ClassicalRegister(k, "readout")
    if t <= 0:
        qc = QuantumCircuit(*qr, cr); qc.measure(qr[n - 1], cr); return qc
    qa = QuantumRegister(t, "trial_a")
    qc = QuantumCircuit(*qr, qa, cr)
    tg = _trial_unitary(n, k)
    data = [q for reg in qr for q in reg[:]]
    for step in range(t):
        targets = [qa[step]] + data
        if step == 0:
            qc.append(tg, targets)
        else:
            qc.append(tg.control(num_ctrl_qubits=step, ctrl_state="0" * step),
                      [qa[j] for j in range(step)] + targets)
    qc.measure(qr[n - 1], cr)
    return qc

classical_golden, classical_meta = build_classical(N, K, T)
coherent_golden = build_coherent(N, K, T)
print(f"classical: {len(classical_golden)} paths, "
      f"{classical_golden[0].num_qubits}q / {classical_golden[0].num_clbits}c each")
print(f"coherent : 1 circuit, {coherent_golden.num_qubits}q / {coherent_golden.num_clbits}c")

## Step 5: Transpile to the backend & compare logical/ISA cost

In [ ]:
def transpile_best(qc, backend, n_seeds, opt_level):
    best, best_d2q = None, None
    for seed in range(n_seeds):
        qt = transpile(qc, backend=backend, optimization_level=opt_level, seed_transpiler=seed)
        d2q = qt.depth(lambda ins: len(ins.qubits) == 2)
        if best is None or d2q < best_d2q:
            best, best_d2q = qt, d2q
    return best

classical_tr = [transpile_best(qc, backend, SEEDS_PER_PATH, OPT_LEVEL) for qc in classical_golden]
coherent_tr  = transpile_best(coherent_golden, backend, SEEDS_PER_PATH, OPT_LEVEL)

def n2q(qc):    return sum(1 for ins in qc.data if len(ins.qubits) == 2)
def d2q(qc):    return qc.depth(lambda ins: len(ins.qubits) == 2)

cost = pd.DataFrame([
    {"flavour": "classical (unrolled)", "circuits": len(classical_tr),
     "2Q gates (total)": sum(n2q(q) for q in classical_tr),
     "2Q depth (max path)": max(d2q(q) for q in classical_tr)},
    {"flavour": "coherent (CSWAP)", "circuits": 1,
     "2Q gates (total)": n2q(coherent_tr),
     "2Q depth (max path)": d2q(coherent_tr)},
])
print(cost.to_string(index=False))

## Step 6: Noise emulation (Pauli twirling) + submission

`λ` is emulated on hardware exactly as in the standard workflow: per twirl instance we sample a
random Pauli string on each data register with probability `λ` and front-load it onto the ISA
circuit. Averaging over `N_TWIRL` instances converges to a `λ`-depolarizing channel. The **same**
twirling is applied to both flavours, so the comparison is apples-to-apples.

Each flavour is submitted as a **single** job (all λ × twirls as separate circuits).

In [ ]:
def twirl(qc_tr, golden, epsilon, k):
    """Return a copy of the transpiled circuit with a sampled Pauli twirl front-loaded (ISA-safe)."""
    inst = qc_tr.copy()
    data_regs = [r for r in golden.qregs if r.name.startswith("R")]
    ops = PauliTwirlingStrategy(k).generate_noise_ops(data_regs, epsilon)
    layout = qc_tr.layout.initial_layout if qc_tr.layout else None
    for gate, logical_q in ops:
        target = None
        if layout is not None and logical_q in layout:
            target = inst.qubits[layout[logical_q]]
        elif logical_q in inst.qubits:
            target = logical_q
        if target is None:
            continue
        if gate.name == "z":
            inst.data.insert(0, CircuitInstruction(RZGate(np.pi), (target,), ()))
        elif gate.name == "y":
            inst.data.insert(0, CircuitInstruction(RZGate(np.pi), (target,), ()))
            inst.data.insert(0, CircuitInstruction(XGate(), (target,), ()))
        else:  # x
            inst.data.insert(0, CircuitInstruction(gate, (target,), ()))
    return inst

def submit_flavour(name, tag):
    """Build all (lambda, twirl[, path]) circuits, submit one job, return (job_id, result, index_map)."""
    pubs, index = [], []   # index[i] = (lambda, path_idx)
    for lam in LAMBDAS:
        for _ in range(N_TWIRL):
            if name == "classical":
                for p in range(len(classical_tr)):
                    pubs.append((twirl(classical_tr[p], classical_golden[p], float(lam), K), None, SHOTS))
                    index.append((float(lam), p))
            else:
                pubs.append((twirl(coherent_tr, coherent_golden, float(lam), K), None, SHOTS))
                index.append((float(lam), 0))
    sampler = IBMSampler(mode=backend)
    sampler.options.environment.job_tags = ["QPA", "cswap_compare", tag]
    job = sampler.run(pubs)
    print(f"  {name}: submitted {len(pubs)} circuits as job {job.job_id()}")
    result = job.result()
    return job.job_id(), result, index

print("Submitting classical job ..."); t0 = time.time()
clf_id, clf_res, clf_idx = submit_flavour("classical", f"classical_n{N}")
print("Submitting coherent job ...")
coh_id, coh_res, coh_idx = submit_flavour("coherent", f"coherent_n{N}")
print(f"Both jobs done in {time.time() - t0:.0f}s wall (incl. queue).")

## Step 7: Fidelity per λ

In [ ]:
rp = ResultProcessor(K)
clf_counts = ResultProcessor.extract_counts_from_job_result(clf_res)
coh_counts = ResultProcessor.extract_counts_from_job_result(coh_res)

def readout_success(counts, k):
    tot = sum(counts.values())
    return (sum(c for b, c in counts.items() if b.replace(" ", "")[-k:] == "0" * k) / tot) if tot else 0.0

fid_classical, fid_coherent = {}, {}
for lam in LAMBDAS:
    lam = float(lam)
    # classical: post-select per path via ResultProcessor (aggregates success/total per path group)
    cc, mm, tcl = [], [], []
    for i, (l, p) in enumerate(clf_idx):
        if l == lam:
            cc.append(clf_counts[i]); mm.append(classical_meta[p]); tcl.append(classical_tr[p].num_clbits)
    fid_classical[lam] = rp.process_unrolled_results(cc, mm, tcl) if cc else float("nan")
    # coherent: direct readout success, pooled over twirls
    pooled = Counter()
    for i, (l, _) in enumerate(coh_idx):
        if l == lam:
            pooled.update(coh_counts[i])
    fid_coherent[lam] = readout_success(pooled, K)

def theory_n3(lam):
    return (1 / 8) * (8 - 2 * lam - 7 * lam ** 2 + 3 * lam ** 3)

df = pd.DataFrame({
    "lambda": [float(l) for l in LAMBDAS],
    "classical": [fid_classical[float(l)] for l in LAMBDAS],
    "coherent":  [fid_coherent[float(l)]  for l in LAMBDAS],
    "theory":    [theory_n3(float(l))     for l in LAMBDAS],
})
df["coherent - classical"] = df["coherent"] - df["classical"]
print(df.to_string(index=False))

with open(os.path.join(RESULTS_DIR, f"compare_n{N}_{backend.name}.json"), "w") as f:
    json.dump({"meta": {"backend": backend.name, "device": DEVICE, "hardware": USE_HARDWARE,
                        "N": N, "K": K, "T": T, "n_twirl": N_TWIRL, "shots": SHOTS,
                        "lambdas": [float(l) for l in LAMBDAS],
                        "classical_job": clf_id, "coherent_job": coh_id},
              "results": df.to_dict("records")}, f, indent=2)
df.to_csv(os.path.join(RESULTS_DIR, f"compare_n{N}_{backend.name}.csv"), index=False)

## Step 8: Plot

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(8, 5))
lam_fine = np.linspace(0, 1, 200)
ax.plot(lam_fine, theory_n3(lam_fine), "--", color="gray", lw=1.3, alpha=0.7, label="theory (N=3, K=2)")
ax.plot(df["lambda"], df["classical"], "o-", color="C0", ms=7, lw=1.8, label="classical (unrolled + post-select)")
ax.plot(df["lambda"], df["coherent"],  "s-", color="C2", ms=7, lw=1.8, label="coherent (CSWAP / Fredkin)")
ax.set_xlabel(r"$\lambda$: per-register depolarizing strength")
ax.set_ylabel(r"$P(\mathrm{readout}=|00\rangle)$")
ax.set_title(f"Classical vs. coherent CSWAP on {backend.name}  (N={N}, K={K}, T={T})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(loc="best")
fig.tight_layout()
out = os.path.join("sandbox", "cswap", f"compare_classical_vs_cswap_{backend.name}.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
print("Saved", out)
plt.show()

## How this affects the standard workflow

`end_to_end_unrolled_pittsburgh_t1.ipynb` is built around the **unrolled** strategy. The coherent
CSWAP approach changes several pillars of that workflow — here is the concrete impact for `N=3, T=1`:

1. **Circuit count collapses `2^((N-1)/2) → 1`.** The standard workflow builds, transpiles
   (`find_best_transpilation_*`), QPY-caches, and submits **one circuit per path**. Coherent needs a
   single circuit, so the whole `transpiled_per_exp[n]` / per-path `summary.json` machinery and the
   `build_noisy_batch` loop-over-paths shrink to one circuit. (See the scaling notebook for how this
   gap widens with `N`.)

2. **No mid-circuit measurement / no `if_test`.** The unrolled paths still measure the Schur ancilla
   mid-circuit and carry post-selection `conditions`; coherent measures **only** the reserve register
   at the end. That removes `ResultProcessor`'s post-selection step (`process_unrolled_results` →
   plain readout success) and side-steps the nested-`if_test` transpiler bug entirely.

3. **Fidelity bookkeeping changes.** Classical fidelity = Σ over path groups of success/total (post-
   selected); coherent fidelity = a single readout-success ratio. The plotting/theory comparison code
   is unchanged (same `theory_curve`), but the per-path aggregation in Steps 5–8 of the standard
   notebook is no longer needed.

4. **Cost moves from *breadth* to *depth*.** Classical keeps each circuit shallow but multiplies their
   number; coherent keeps one circuit but pays for the Fredkin routing (and, for `T>1`, multi-
   controlled trial unitaries). The Step-5 table above shows the `N=3` trade today; whether coherent
   *wins on fidelity* on real hardware is exactly what the plot answers.

5. **Twirling/DD are reusable.** The Pauli-twirl noise emulation and DD sweep transfer unchanged — the
   twirl is front-loaded on the data registers either way — so the DD-on/off comparison in the standard
   notebook would slot straight onto the coherent circuit.

**Bottom line:** if coherent CSWAP holds fidelity at `N=3` (the plot above), it removes the path-
explosion, the QPY cache, and the post-selection logic from the workflow — at the price of one deeper
circuit. The scaling notebook (`cswap_scaling.ipynb`) shows where that trade flips against you.